In [ ]:
%config InlineBackend.figure_format = 'retina'
import os
import sys
import importlib
from pathlib import Path
root_path = Path(os.getcwd())
sys.path.append(str(Path('..').resolve()))

import math
import polars as pd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from loader import load_player_data_for_scoring
from aggregation import aggregate_all
from steps.filter import filter_all, filtering_summary
from steps.decay import apply_decay, decay_summary
from steps.normalization import normalize_all, normalization_summary

STATSBOMB_DIR = Path("..") / "data" / "Statsbomb"

In [2]:
raw = load_player_data_for_scoring("recent_club_players")
clean = aggregate_all(raw, verbose=False)


Loading: recent_club_players
Seasons: ['2021/2022', '2022/2023', '2023/2024']

  ── 2021/2022 ──
    ✓ xg__player__totals.csv — 293 rows, 284 players
    ✓ progression__player__profile.csv — 1,175 rows, 1123 players
    ✓ advanced__player__packing.csv — 5,104 rows, 1053 players
    ✓ advanced__player__xg_chain.csv — 5,403 rows, 1151 players
    ✓ advanced__player__xg_buildup.csv — 4,731 rows, 1083 players
    ✓ advanced__player__network_centrality.csv — 6,194 rows, 1244 players
    ✓ defensive__player__profile.csv — 1,919 rows, 650 players
    ✓ defensive__player__pressures.csv — 5,647 rows, 1167 players

  ── 2022/2023 ──
    ✓ xg__player__totals.csv — 134 rows, 131 players
    ✓ progression__player__profile.csv — 884 rows, 846 players
    ✓ advanced__player__packing.csv — 2,417 rows, 874 players
    ✓ advanced__player__xg_chain.csv — 2,421 rows, 888 players
    ✓ advanced__player__xg_buildup.csv — 2,125 rows, 822 players
    ✓ advanced__player__network_centrality.csv — 2,740 rows, 9

In [ ]:
filtered = filter_all(clean, verbose=True)
filtering_summary(filtered)


STEP 2: FILTERING
Hard threshold: 270 min  |  Shrinkage floor: 180 min
  [filter] xg__player__totals.csv:
           filter_col=matches, threshold=3, floor=2
           611 → 599 rows kept (12 dropped), 82 flagged for shrinkage
  [filter] progression__player__profile.csv:
           filter_col=minutes_played, threshold=270, floor=180
           3,113 → 1,524 rows kept (1,589 dropped), 512 flagged for shrinkage
  [filter] advanced__player__packing.csv:
           filter_col=total_passes, threshold=30, floor=20
           2,985 → 2,086 rows kept (899 dropped), 384 flagged for shrinkage
  [filter] advanced__player__xg_chain.csv:
           filter_col=minutes_played, threshold=270, floor=180
           3,226 → 1,422 rows kept (1,804 dropped), 481 flagged for shrinkage
  [filter] advanced__player__xg_buildup.csv:
           filter_col=minutes_played, threshold=270, floor=180
           3,011 → 1,280 rows kept (1,731 dropped), 424 flagged for shrinkage
  [filter] advanced__player__network_c

In [14]:
decayed = apply_decay(filtered, verbose=True)
decay_summary(decayed)


STEP 3: TIME DECAY WEIGHTING
Weights: {'2023/2024': 1.0, '2022/2023': 0.75, '2021/2022': 0.5}
  [decay] xg__player__totals.csv:
           2021/2022: 185 rows × 0.5
           2022: 104 rows × 0.5
           2022/2023: 28 rows × 0.75
           2023: 100 rows × 0.75
           2023/2024: 27 rows × 1.0
           2024: 155 rows × 1.0
  [decay] progression__player__profile.csv:
           2021/2022: 647 rows × 0.5
           2022/2023: 360 rows × 0.75
           2023/2024: 517 rows × 1.0
  [decay] advanced__player__packing.csv:
           2021/2022: 453 rows × 0.5
           2022: 336 rows × 0.5
           2022/2023: 256 rows × 0.75
           2023: 447 rows × 0.75
           2023/2024: 119 rows × 1.0
           2024: 475 rows × 1.0
  [decay] advanced__player__xg_chain.csv:
           2021/2022: 290 rows × 0.5
           2022: 324 rows × 0.5
           2022/2023: 61 rows × 0.75
           2023: 278 rows × 0.75
           2023/2024: 68 rows × 1.0
           2024: 401 rows × 1.0
  [decay]

In [16]:
import polars as pl

for filename, df in decayed.items():
    metrics_in_file = [
        (k, v["column"]) 
        for k, v in __import__('player_score.player_metrics_config', fromlist=['PLAYER_METRICS']).PLAYER_METRICS.items()
        if v["file"] == filename and v["column"] in df.columns
    ]
    for metric_key, col in metrics_in_file:
        stats = df[col].drop_nulls()
        print(f"{metric_key}: min={stats.min():.3f}, median={stats.median():.3f}, max={stats.max():.3f}, skew={stats.skew():.2f}")

finishing_quality: min=-3.260, median=-0.170, max=9.560, skew=2.00
xg_volume: min=0.070, median=0.880, max=23.300, skew=4.98
progressive_passes: min=0.000, median=1.760, max=8.950, skew=1.12
progressive_carries: min=0.000, median=1.155, max=10.840, skew=1.50
packing: min=0.000, median=0.000, max=2.467, skew=1.16
xg_chain: min=0.009, median=0.422, max=1.814, skew=1.07
team_involvement: min=3.175, median=36.130, max=97.559, skew=0.45
xg_buildup: min=0.007, median=0.305, max=1.849, skew=1.29
network_centrality: min=2.241, median=16.362, max=36.332, skew=0.11
defensive_actions: min=27.000, median=63.000, max=639.000, skew=2.83
high_turnovers: min=0.000, median=5.000, max=87.000, skew=3.11
pressure_volume: min=0.796, median=12.177, max=40.275, skew=0.70
pressure_success: min=0.000, median=25.808, max=79.575, skew=0.59


In [ ]:
normalized = normalize_all(decayed, verbose=True)
normalization_summary(normalized)


STEP 4: NORMALIZATION
Method: per-season | log/log1p → rank | rank | zscore


NameError: name 'math' is not defined